# Hugging Face Applications — Lesson 5: Text Classification

> Learning material for **Hugging Face Applications**. Companion to the lesson script `05_Text_Classification.py` (same content, runnable without Jupyter).

**Task ID:** HF-205  |  **Folder:** `documentation`


## What is text classification?

**Classification** = putting an input into one of a *fixed set* of categories. For text:

> 💬 *"I absolutely loved the film"*  →  🏷️ **POSITIVE** (score 0.99)

Each prediction comes with a **score** (0..1) — the model's confidence.

## Single-label vs. multi-label

- **Single-label** (this lesson): exactly one category per text. Scores sum to 1.
- **Multi-label**: several categories can be true at once (a movie can be both *funny* and *romantic*). Uses a dedicated model.

## How a classifier works

DistilBERT reads the text, summarizes it into one special vector (`[CLS]`), and a small final layer turns that vector into one score per category (softmax → probabilities).

> **Analogy:** reading the text, forming an opinion, then rating it 0–100 for each category.

The pipeline handles everything:


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)
# first call downloads the model (~270 MB) and prints progress bars


**Classify one text:**

In [ ]:
r = classifier("I absolutely loved the film, the acting was brilliant.")
print(r)   # [{'label': 'POSITIVE', 'score': 0.99...}]


## Batch: classify many texts in one call

Pass a **list**. The pipeline tokenizes and runs them together (batched on GPU; on CPU it still beats a Python loop):


In [ ]:
texts = [
    "The service was slow and the food was cold.",
    "The product works exactly as described, very happy.",
    "I would not recommend this to anyone.",
    "It was okay, nothing special but nothing terrible either.",
]
results = classifier(texts)
for text, r in zip(texts, results):
    print(f"{r['label']:<9} ({r['score']:.2f})  {text!r}")


## top_k: see more of the probability distribution

With `top_k=2` you get each text's top-2 categories and scores — useful
to see how close the runner-up was:


In [ ]:
results = classifier("It was fine, I guess.", top_k=2)
for r in results:
    print(f"{r['label']:<9} {r['score']:.3f}")


## Try it yourself

1. Classify 10 of your own sentences (reviews, tweets, notes).
2. Find a case where the score is ~0.5 (uncertain) — why is it uncertain?
3. Swap in a different classifier: `cardiffnlp/twitter-roberta-base-sentiment-latest` (3 classes) or `j-hartmann/emotion-english-distilroberta-base` (7 emotions).
4. **Multi-label bonus:** try a model like `facebook/bart-large-mnli` with `top_k=None` — that returns ALL classes.

## Common pitfalls

- Labels differ per model (`POSITIVE/NEGATIVE`, `pos/neg/neu`, ...) — always print one result first.
- `top_k` semantics vary pre- and post-v4.20; with a single text it returns a list when `top_k>1`.

## Summary

- Classification = pick a category from a fixed set; score = confidence.
- Batch with a list; read `label` + `score` from each result.

**Next lesson:** HF-206 — Sentiment Analysis.  |  Extra reading: `../resources/reference_links.md`
